# 04 — Review: segmentation + canonicalization interleaved

Pick a protocol by id and render an RTL view of the full pipeline output:
**segment 1 (raw utterances) → canonical 1 → segment 2 → canonical 2 → ...**

- Raw block (gray): every utterance of the segment, with ⚠ marks for mid-utterance matter changes.
- Canonical block (green): subject, decision, amounts, length ratio, and the neutral 3rd-person rewrite.
- Missing canonicalization → red strip (run notebook 03); missing segmentation → clear error (run notebook 02).

No GPU needed — this only reads `data/segmentations/` + `data/canonicalizations/` from Drive.
The rendered page is also saved to `eval/review_{file_id}.html`.

In [ ]:
# === Cell 1: Drive mount + configuration ===
import json
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

BASE          = Path('/content/drive/MyDrive/NLP ADVANCED/FinalProject/NLP_ADV')
PROTOCOLS_DIR = BASE / 'data' / 'protocols'
SEGMENTS_DIR  = BASE / 'data' / 'segmentations'
CANON_DIR     = BASE / 'data' / 'canonicalizations'
EVAL_DIR      = BASE / 'eval'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

assert PROTOCOLS_DIR.exists(), f'missing {PROTOCOLS_DIR} - check the Drive layout'
assert SEGMENTS_DIR.exists(),  f'missing {SEGMENTS_DIR} - run notebook 02 first'

INDEX = json.loads((PROTOCOLS_DIR / 'protocols_index.json').read_text(encoding='utf-8'))
print('protocols     :', len(INDEX))
print('segmentations :', len(list(SEGMENTS_DIR.glob('*.segments.json'))))
print('canonical     :', len(list(CANON_DIR.glob('*.canonical.json'))) if CANON_DIR.exists() else 0)

Mounted at /content/drive
protocols     : 927
segmentations : 20
canonical     : 13


In [ ]:
# === Cell 2: helpers (id resolution + HTML builder) ===
import html

def resolve_file_id(query):
    """query = seq prefix ('000004'), full file_id, or doc_id ('25_ptv_...')."""
    for e in INDEX:
        if e['file_id'] == query or e['doc_id'] == query:
            return e['file_id']
    matches = [e['file_id'] for e in INDEX if e['file_id'].startswith(query)]
    if len(matches) == 1:
        return matches[0]
    raise ValueError(f'no protocol matches {query!r}' if not matches
                     else f'{query!r} is ambiguous: {matches}')

def esc(s):
    return html.escape(str(s or ''))

def build_page(protocol, seg_result, canon_result, max_chars=600):
    utts = {str(u['id']): u for u in protocol['utterances']}
    canon_by_seg = {s['seg_id']: s for s in (canon_result or {}).get('segments', [])}
    splits = {str(s['utterance_id']): s for s in seg_result.get('intra_utterance_splits', [])}

    canon_note = (f"canonical: {len(canon_by_seg)}/{seg_result['n_segments']}"
                  if canon_result else 'canonical: MISSING (run notebook 03)')
    head = (f"{seg_result['file_id']} | {seg_result['date'][:10]} | "
            f"{seg_result['n_segments']} segments | {canon_note}")

    out = ["<style>body{background:#fafafa}</style><meta charset='utf-8'>",
           "<div dir='rtl' style='font-family:sans-serif;max-width:960px;margin:auto;"
           "background:#fafafa;color:#111;padding:8px 16px'>",
           f"<h2 dir='ltr' style='text-align:center;color:#111'>{esc(head)}</h2>",
           f"<p style='color:#555'>{esc(protocol.get('title', ''))}</p>"]

    for seg in seg_result['segments']:
        sid = seg['seg_id']
        out.append("<div style='background:#f5f5f0;color:#111;padding:12px;margin:14px 0 0;"
                   "border-radius:8px 8px 0 0;border:1px solid #ddd'>")
        out.append(f"<b>— {esc(sid)} [{seg['start']}–{seg['end']}] · "
                   f"{seg['n_utterances']} utterances —</b>")
        for uid in seg['utterance_ids']:
            u = utts[str(uid)]
            txt = u['text'].strip()
            if max_chars and len(txt) > max_chars:
                txt = txt[:max_chars] + ' …'
            mark = ''
            if str(uid) in splits:
                s = splits[str(uid)]
                flag = '' if s.get('anchor_found') else ' (anchor not located!)'
                mark = (f"<div style='color:#993c1d;font-size:13px'>⚠ matter changes inside "
                        f"this utterance{esc(flag)}, at: «{esc(s['starts_with'])}» → "
                        f"{esc(s['new_matter'])}</div>")
            out.append(f"<p style='margin:6px 0'><b>[{uid}] {esc(u['speaker'])}:</b> "
                       f"{esc(txt)}{mark}</p>")
        out.append('</div>')

        c = canon_by_seg.get(sid)
        if c:
            meta = ' · '.join(x for x in [
                esc(c.get('decision')),
                esc(', '.join(map(str, c.get('amounts') or []))),
                f"raw {c.get('n_chars_raw')} → canonical {c.get('n_chars_canonical')} chars "
                f"(x{c.get('ratio')})"] if x)
            out.append("<div style='background:#e8f3e8;color:#111;padding:12px;margin:0 0 14px;"
                       "border-radius:0 0 8px 8px;border:1px solid #cde3cd;border-top:0'>")
            out.append(f"<b>✎ קנוניזציה — {esc(c.get('subject'))}</b>"
                       f"<div style='color:#555;font-size:13px'>{meta}</div>")
            out.append(f"<p style='margin:8px 0 0'>{esc(c.get('canonical'))}</p>")
            out.append('</div>')
        else:
            out.append("<div style='background:#fdf0f0;padding:8px 12px;margin:0 0 14px;"
                       "border-radius:0 0 8px 8px;border:1px solid #eecccc;border-top:0;"
                       "color:#993c1d'>✗ אין קנוניזציה למקטע זה</div>")

    out.append('</div>')
    return '\n'.join(out)

In [ ]:
# === Cell 3: pick a protocol and render ===
from IPython.display import HTML, display

PROTOCOL_ID = '000004'   # seq prefix, full file_id, or doc_id
MAX_CHARS   = 0          # 0 = show full utterance text; set e.g. 600 to truncate long ones

file_id  = resolve_file_id(PROTOCOL_ID)
protocol = json.loads((PROTOCOLS_DIR / f'{file_id}.json').read_text(encoding='utf-8'))

seg_fp = SEGMENTS_DIR / f'{file_id}.segments.json'
assert seg_fp.exists(), f'no segmentation for {file_id} - run notebook 02 on it first'
seg_result = json.loads(seg_fp.read_text(encoding='utf-8'))

canon_fp = CANON_DIR / f'{file_id}.canonical.json'
canon_result = json.loads(canon_fp.read_text(encoding='utf-8')) if canon_fp.exists() else None
canon_by_seg = {s['seg_id']: s for s in (canon_result or {}).get('segments', [])}

print(f"{file_id} | {seg_result['date'][:10]} | {seg_result['n_segments']} segments"
      f" | canonical for {len(canon_by_seg)}/{seg_result['n_segments']}")
for seg in seg_result['segments']:
    c = canon_by_seg.get(seg['seg_id'])
    label = (c.get('subject') or '(no subject)') if c else '✗ no canonicalization'
    print(f"  {seg['seg_id']}  [{seg['start']:>4}-{seg['end']:>4}] "
          f"{seg['n_utterances']:>3} utts | {label}")

page = build_page(protocol, seg_result, canon_result, max_chars=MAX_CHARS)
out_fp = EVAL_DIR / f'review_{file_id}.html'
out_fp.write_text(page, encoding='utf-8')
print('\nsaved ->', out_fp)
display(HTML(page))

000004_2022-11-29 | 2022-11-29 | 42 segments | canonical for 42/42
  000004_2022-11-29_s001  [   1-   9]   9 utts | דיון בנושא יוקר הדיור והמשכנתאות
  000004_2022-11-29_s002  [  10-  12]   3 utts | טיפול בלקוחות מוגבלים ובשוק האפור
  000004_2022-11-29_s003  [  13-  14]   2 utts | פתיחת הדיון בוועדת הכספים
  000004_2022-11-29_s004  [  15-  18]   4 utts | פתרונות למשבר הדיור עבור זוגות צעירים
  000004_2022-11-29_s005  [  19-  19]   1 utts | דיון בהעברות תקציביות ועודפים של משרד השיכון
  000004_2022-11-29_s006  [  20-  45]  26 utts | אישור עודפי תקציב מטה משרד השיכון
  000004_2022-11-29_s007  [  46-  56]  11 utts | בקשת העברה לתקציב פיתוח משרד השיכון
  000004_2022-11-29_s008  [  57-  62]   6 utts | מנגנון תקצוב הסכמי גג ופיתוח
  000004_2022-11-29_s009  [  63-  65]   3 utts | פעולות תומכות בבנייה למגורים
  000004_2022-11-29_s010  [  66-  67]   2 utts | הגדרת פעולות שיכון שוטפות
  000004_2022-11-29_s011  [  68-  72]   5 utts | תיאום תשתיות וכבישי גישה לדיור ציבורי
  000004_2022-11-29_s012  